# 03 GRPO / RLVR

Purpose: run verifier-reward GRPO/RLVR from the configured policy checkpoint and compare validation-selected checkpoints on held-out FinChain test data. This public run initializes from the selected SFT checkpoint.

Expected inputs: selected SFT checkpoint from `shannan-liu1/qwen25-1p5b-finchain-v2-sft-selected`, FinChain template-disjoint splits, and GRPO runtime settings mirrored in `configs/finchain/grpo/qwen25_1_5b_grpo.yaml`.

Expected outputs: GRPO/RLVR candidate checkpoints, validation/test eval summaries, and selected model artifacts.

HF status: GRPO/RLVR selected/candidate repos are listed in `docs/hf_checkpoints.md`.


## Before you run a single cell in this notebook - terminal pre-flight

This notebook runs Group Relative Policy Optimization (GRPO) on FinChain verifier rewards. Run this terminal block first on a fresh GPU environment. It pins the CUDA/PyTorch stack before NCCL work, keeps HF cache on the persistent volume, requires the FinChain v2 template-disjoint JSONLs and manifest under the local data path, and logs into W&B/Hugging Face before training starts. If you are restarting a stopped environment or recovering from a package failure, rerun this block before resuming training; do not jump straight to an Accelerate launch.

```bash
cd /workspace
test -d finpost || git clone https://github.com/shannan-liu1/finpost.git
cd /workspace/finpost
git checkout main
git pull --ff-only

# Keep package caches and temp files on the persistent volume, not the small container disk.
export HF_HOME=/workspace/hf-cache
export PIP_CACHE_DIR=/workspace/pip-cache
export TMPDIR=/workspace/tmp
export WANDB_DIR=/workspace/wandb
mkdir -p "$HF_HOME" "$PIP_CACHE_DIR" "$TMPDIR" "$WANDB_DIR"

# Install project deps, then fail fast on CUDA/NCCL drift. If the guard fails,
# the repair script removes CUDA 13 pip packages and reinstalls the A40-safe
# Torch CUDA 12.4 stack:
#   torch==2.6.0+cu124
# The repair script also removes optional torchvision/torchaudio wheels;
# finpost does not use them, and broken optional wheels can make transformers imports fail.
python -m pip install -e ".[dev,rlvr,chaineval]"
nvidia-smi || true
bash scripts/repair_cuda_stack.sh
python -m pip install -e ".[dev,rlvr,chaineval]"
python scripts/check_cuda_stack.py

# Persistent cache + FinChain split paths. The public repo does not
# track these JSONLs; generate or place them under data/finchain_v2_template_disjoint
# before the paid GPU run.
export HF_HOME=/workspace/hf-cache
export PIP_CACHE_DIR=${PIP_CACHE_DIR:-/workspace/pip-cache}
export TMPDIR=${TMPDIR:-/workspace/tmp}
export WANDB_DIR=${WANDB_DIR:-/workspace/wandb}
export WANDB_MODE=${WANDB_MODE:-online}
export WANDB_PROJECT=finpost-finchain-grpo
mkdir -p /workspace/data/finchain_v2_template_disjoint "$HF_HOME" "$PIP_CACHE_DIR" "$TMPDIR" "$WANDB_DIR"
for f in train.jsonl validation.jsonl test.jsonl manifest.json; do
  test -f "data/finchain_v2_template_disjoint/$f" || { echo "Missing data/finchain_v2_template_disjoint/$f; generate or place the FinChain v2 splits before training." >&2; exit 1; }
  cp -n "data/finchain_v2_template_disjoint/$f" /workspace/data/finchain_v2_template_disjoint/
done
export FINPOST_FINCHAIN_TRAIN_JSONL=/workspace/data/finchain_v2_template_disjoint/train.jsonl
export FINPOST_FINCHAIN_VALIDATION_JSONL=/workspace/data/finchain_v2_template_disjoint/validation.jsonl
export FINPOST_FINCHAIN_TEST_JSONL=/workspace/data/finchain_v2_template_disjoint/test.jsonl
python scripts/audit_finchain_template_disjoint_manifest.py --data-dir data/finchain_v2_template_disjoint --out artifacts/preflight/finchain_v2_manifest_audit.json
python scripts/gpu_preflight.py --out artifacts/preflight/preflight_report.json --timeout-sec 900

# Auth. `wandb status` should show your username. `huggingface-cli whoami`
# should succeed if you plan to push checkpoints or access private/gated repos.
wandb login
wandb status
huggingface-cli login
huggingface-cli whoami

# Pre-download model snapshots so an Accelerate launch does not look hung while
# it is only fetching weights.
HF_HOME=/workspace/hf-cache python scripts/warm_hf_cache.py Qwen/Qwen2.5-0.5B Qwen/Qwen2.5-1.5B
```

Distributed rule of thumb: record distributed training only for cells that actually use `accelerate launch --num_processes 2` or explicit pair-generation sharding. `NCCL_P2P_DISABLE=1 NCCL_IB_DISABLE=1` is the safe 2x A40 starting point; remove those only after a small distributed canary passes without the NCCL/CUDA-driver error.


# FinChain GRPO GPU Notebook

GRPO is the primary RLVR method evaluated here: sample K completions per prompt, score each with the FinChain verifier, normalize rewards within the group, update the policy with KL control. This notebook uses TRL (`scripts/train_finchain_trl_grpo.py`) for the training path and keeps the repo math in `src/finpost/training/grpo.py` as the readable reference.

## Step 1 - Sanity-check the environment

In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib
import json
import os
import platform
import subprocess
import sys
import time

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = Path("/workspace/finpost") if Path("/workspace/finpost").exists() else PROJECT_ROOT
os.chdir(PROJECT_ROOT)

RESULTS_DIR = PROJECT_ROOT / "results" / "finchain_grpo"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def progress(title, detail=None):
    stamp = time.strftime("%H:%M:%S")
    print(f"[{stamp}] {title}")
    if detail:
        print(detail)


def run_cmd(cmd, *, check=False):
    progress("running command", cmd)
    completed = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f"command failed with exit {completed.returncode}: {cmd}")
    return completed


def append_cost_event(stage, **payload):
    path = RESULTS_DIR / "cost_ledger.jsonl"
    row = {"stage": stage, "time": time.strftime("%Y-%m-%dT%H:%M:%S"), **payload}
    with path.open("a", encoding="utf-8") as fp:
        fp.write(json.dumps(row, sort_keys=True) + "\n")
    print(json.dumps(row, indent=2, sort_keys=True))
    return row


progress("project root", str(PROJECT_ROOT))
progress("python", sys.version.split()[0])
progress("platform", platform.platform())

## Distributed launch preflight

Run this after the setup cell and before any multi-GPU command. If it reports fewer than 2 CUDA devices, stay on the single-GPU path. See `notebooks/01_sft_ablation.ipynb` for the full environment checklist.

In [ ]:
progress("distributed launch preflight")
try:
    import accelerate
    print("accelerate:", accelerate.__version__)
except Exception as exc:
    print("accelerate import failed:", repr(exc))

try:
    import torch
    print("cuda devices:", torch.cuda.device_count())
    for idx in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(idx)
        print(f"gpu {idx}: {props.name}, vram={props.total_memory / 1e9:.1f} GB")
except Exception as exc:
    print("torch cuda check failed:", repr(exc))

for key in ["CUDA_VISIBLE_DEVICES", "WORLD_SIZE", "LOCAL_RANK", "RANK"]:
    print(f"{key}={os.environ.get(key)}")

run_cmd("accelerate env")


## GPU setup guardrails

See `notebooks/01_sft_ablation.ipynb` for the full checklist. GRPO-specific note: `accelerate launch scripts/train_finchain_trl_grpo.py` starts one process per GPU; TRL handles rollout generation, optimizer sync, and checkpoint writes.

In [ ]:
import os
import shutil
from pathlib import Path

os.environ.setdefault("HF_HOME", "/workspace/hf-cache")
os.environ.setdefault("PIP_CACHE_DIR", "/workspace/pip-cache")
os.environ.setdefault("TMPDIR", "/workspace/tmp")
os.environ.setdefault("WANDB_DIR", "/workspace/wandb")
os.environ.setdefault("WANDB_MODE", "online")
os.environ.setdefault("WANDB_PROJECT", "finpost-finchain-grpo")

Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["PIP_CACHE_DIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["TMPDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["WANDB_DIR"]).mkdir(parents=True, exist_ok=True)

runtime_data_dir = Path("/workspace/data/finchain_v2_template_disjoint")
repo_data_dir = PROJECT_ROOT / "data" / "finchain_v2_template_disjoint"
runtime_data_dir.mkdir(parents=True, exist_ok=True)

missing_inputs = []
for split in ["train", "validation", "test"]:
    source = repo_data_dir / f"{split}.jsonl"
    target = runtime_data_dir / f"{split}.jsonl"
    if not target.exists() and source.exists():
        shutil.copy2(source, target)
    os.environ[f"FINPOST_FINCHAIN_{split.upper()}_JSONL"] = str(target)
    if not target.exists():
        missing_inputs.append(str(target))

manifest_source = repo_data_dir / "manifest.json"
manifest_target = runtime_data_dir / "manifest.json"
if not manifest_target.exists() and manifest_source.exists():
    shutil.copy2(manifest_source, manifest_target)
if not manifest_target.exists():
    missing_inputs.append(str(manifest_target))

if missing_inputs:
    raise FileNotFoundError(
        "Missing FinChain v2 split JSONLs or manifest. Generate or place them under "
        "data/finchain_v2_template_disjoint/ before training: "
        + ", ".join(missing_inputs)
    )

progress("GPU CUDA/data guard")
run_cmd("python scripts/check_cuda_stack.py", check=True)

progress("GRPO TRL dependency guard")
run_cmd("python scripts/train_finchain_trl_grpo.py --help", check=True)
run_cmd("""
python - <<'PY'
import argparse
from trl import GRPOConfig
from scripts.train_finchain_trl_grpo import _config_kwargs

args = argparse.Namespace(
    output_dir="artifacts/preflight/grpo_config_smoke",
    max_steps=1,
    learning_rate=5e-7,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_generations=2,
    max_completion_length=64,
    temperature=0.8,
    top_p=0.95,
    beta=0.02,
    save_steps=50,
    logging_steps=5,
    report_to="none",
    run_name="grpo-config-smoke",
    use_vllm=False,
    vllm_gpu_memory_utilization=0.3,
    seed=42,
)
GRPOConfig(**_config_kwargs(args, config_cls=GRPOConfig))
print("GRPOConfig OK")
PY
""", check=True)

progress("auth status")
run_cmd("wandb status", check=False)
run_cmd("huggingface-cli whoami", check=False)

for key in [
    "HF_HOME",
    "WANDB_MODE",
    "WANDB_PROJECT",
    "FINPOST_FINCHAIN_TRAIN_JSONL",
    "FINPOST_FINCHAIN_VALIDATION_JSONL",
    "FINPOST_FINCHAIN_TEST_JSONL",
]:
    print(f"{key}={os.environ.get(key)}")


## Step 1.5 - Hugging Face cache warmup

Run once per environment/volume. This downloads tokenizer/config/safetensors into `HF_HOME` without loading the model on GPU. It makes later stalls easier to diagnose: after this cell, a long pause is trainer setup or generation, not model download.


In [ ]:
#HF_WARMUP_MODELS = ['Qwen/Qwen2.5-0.5B', 'Qwen/Qwen2.5-1.5B']
#progress("HF cache warmup", " ".join(HF_WARMUP_MODELS))
#warm_cmd = "HF_HOME=/workspace/hf-cache python scripts/warm_hf_cache.py " + " ".join(HF_WARMUP_MODELS)
#run_cmd(warm_cmd, check=True)


In [ ]:
HF_WARMUP_MODELS = ["Qwen/Qwen2.5-0.5B"]
progress("HF cache warmup", " ".join(HF_WARMUP_MODELS))
warm_cmd = "HF_HOME=/workspace/hf-cache python scripts/warm_hf_cache.py " + " ".join(HF_WARMUP_MODELS)
run_cmd(warm_cmd, check=True)

In [ ]:
progress("GPU preflight")
try:
    import torch
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        for idx in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(idx)
            print(f"gpu {idx}: {props.name}, vram={props.total_memory / 1e9:.1f} GB")
except Exception as exc:
    print("torch check failed:", repr(exc))

run_cmd("nvidia-smi")

In [ ]:
# Confirm core modules import. Verifier roundtrip sanity-check before any GPU work.
for module_name in [
    "finpost.training.grpo",
    "finpost.training.finchain_rlvr",
    "finpost.training.logprobs",
    "finpost.data.finchain_dataset",
    "finpost.evals.finchain_metrics",
]:
    try:
        importlib.import_module(module_name)
        print(module_name, "OK")
    except Exception as exc:
        print(module_name, "FAILED:", repr(exc))

from finpost.data.finchain_dataset import load_finchain, resolve_finchain_path

split_examples = {}
for split in ["train", "validation", "test"]:
    path = resolve_finchain_path(split)
    examples = load_finchain(split)
    split_examples[split] = examples
    print(split, "->", path, "exists=", path.exists(), "rows=", len(examples))

assert len(split_examples["validation"]) == 870, "expected FinChain validation split to contain 870 rows"
assert len(split_examples["test"]) == 870, "expected FinChain test split to contain 870 rows"
train_examples = split_examples["train"]
print("train examples:", len(train_examples))
print("validation examples:", len(split_examples["validation"]))
print("test examples:", len(split_examples["test"]))
print("first prompt id:", train_examples[0].id)
print("first gold answer:", train_examples[0].final_answer)


## Step 2 - Hyperparameters

The reported GRPO/RLVR run samples eight completions per prompt with KL beta ~0.04, group-relative advantages, and the FinChain binary reward. The public run shape is mirrored in `configs/finchain/grpo/qwen25_1_5b_grpo.yaml`; this cell stays inline so the notebook remains self-contained in a fresh GPU environment. Set `train_n` and `max_steps` for the planned run budget.

In [ ]:
GRPO_CONFIG = {
    "base_model_id": "Qwen/Qwen2.5-1.5B",
    "fast_canary_model_id": "Qwen/Qwen2.5-0.5B",
    "policy_hf_dir": "results/checkpoints/qwen25-1p5b-finchain-v2-trl-sft-2gpu/selected",
    "save_dir": "results/checkpoints/qwen25-1p5b-finchain-v2-grpo",
    "hf_selected_repo": "shannan-liu1/qwen25-1p5b-finchain-v2-grpo-selected",
    "hf_candidates_repo": "shannan-liu1/qwen25-1p5b-finchain-v2-grpo-candidates",
    "all_checkpoints_hf_repo": "shannan-liu1/qwen25-1p5b-finchain-v2-grpo-candidates",
    # Rollouts
    "num_generations": 8,
    "max_completion_length": 512,  # generated tokens, not SFT total length
    # Training
    "train_n": 2320,
    "max_steps": 500,
    "save_steps": 100,
    "grpo_eval_steps": [300, 400, 500],
    "per_device_train_batch_size": 8,
    "gradient_accumulation_steps": 4,
    "learning_rate": 5.0e-6,
    "beta_kl": 0.04,
    "wandb_project": "finpost-finchain-grpo",
    "run_name": "qwen25-1p5b-finchain-v2-grpo",
    "reset_outputs_before_rerun": True,
}
print(json.dumps(GRPO_CONFIG, indent=2))

## Step 2.5 - Restore configured GRPO policy checkpoint

Run this on a fresh environment before the canary or full run. It restores the configured policy checkpoint from Hugging Face into the local path used by this notebook, and skips the download when weights already exist. The public run uses the validation-selected SFT checkpoint.


In [ ]:
from huggingface_hub import snapshot_download

SFT_REPO_ID = "shannan-liu1/qwen25-1p5b-finchain-v2-sft-selected"
policy_local_dir = Path(GRPO_CONFIG["policy_hf_dir"])
SFT_ALLOW_PATTERNS = [
    "config.json",
    "generation_config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "added_tokens.json",
    "chat_template.jinja",
    "*.safetensors",
    "*.safetensors.index.json",
    "pytorch_model*.bin",
    "pytorch_model.bin.index.json",
    "vocab.*",
    "merges.txt",
    "*.model",
]

model_files = sorted(policy_local_dir.glob("*.safetensors")) + sorted(policy_local_dir.glob("pytorch_model*.bin"))
if model_files:
    print("GRPO policy checkpoint already present:", policy_local_dir)
else:
    print("Restoring GRPO policy checkpoint from HF:", SFT_REPO_ID, "->", policy_local_dir)
    policy_local_dir.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=SFT_REPO_ID,
        repo_type="model",
        local_dir=str(policy_local_dir),
        allow_patterns=SFT_ALLOW_PATTERNS,
    )
    model_files = sorted(policy_local_dir.glob("*.safetensors")) + sorted(policy_local_dir.glob("pytorch_model*.bin"))
    if not model_files:
        raise RuntimeError(f"Downloaded GRPO policy checkpoint has no model weights: {policy_local_dir}")

print("policy model files:", [path.name for path in model_files])


## Step 3 - Fast GRPO canary (0.5B, 5 steps)

Catches: TRL GRPO config validation failure, reward function returns NaN, generation path broken, CUDA/NCCL instability. This uses `Qwen/Qwen2.5-0.5B` with tiny rollouts for a cheap stack check. The full GRPO run below uses the configured 1.5B policy checkpoint.


In [ ]:
canary_out = f"{GRPO_CONFIG['save_dir']}-canary-0p5b"
canary_model = GRPO_CONFIG["fast_canary_model_id"]
canary_cmd = (
    "HF_HOME=/workspace/hf-cache python scripts/train_finchain_trl_grpo.py "
    f"--model {canary_model} "
    "--train-n 64 "
    "--max-steps 5 "
    "--num-generations 2 "
    "--max-completion-length 96 "
    "--per-device-train-batch-size 2 "
    "--gradient-accumulation-steps 2 "
    f"--learning-rate {GRPO_CONFIG['learning_rate']} "
    f"--beta {GRPO_CONFIG['beta_kl']} "
    f"--output-dir {canary_out} "
    "--save-steps 999 --logging-steps 1 --report-to none"
)
canary_start = time.perf_counter()
canary_result = run_cmd(canary_cmd, check=False)
append_cost_event(
    "grpo_canary_fast_0p5b",
    elapsed_sec=round(time.perf_counter() - canary_start, 1),
    exit_code=canary_result.returncode,
    out_dir=canary_out,
    model=canary_model,
)
if canary_result.returncode != 0:
    raise RuntimeError("GRPO canary failed. Do not run the full training.")


## Step 4 - Full GRPO training run

Default path: two-GPU TRL GRPO via `accelerate launch --num_processes 2` when two CUDA devices are visible. The fallback path is single-GPU TRL GRPO only when the environment exposes fewer than two GPUs. Keep `num_generations` compatible with the effective batch constraints from the canary before scaling.


In [ ]:
import shutil
import torch as _t

if GRPO_CONFIG["reset_outputs_before_rerun"]:
    for reset_path in [
        Path(GRPO_CONFIG["save_dir"]),
        PROJECT_ROOT / "results" / "evals" / "grpo_v2_template_disjoint",
    ]:
        if reset_path.exists():
            print("removing stale artifact:", reset_path)
            shutil.rmtree(reset_path)

gpu_count = _t.cuda.device_count() if _t.cuda.is_available() else 0
full_per_device_train_batch_size = GRPO_CONFIG['per_device_train_batch_size'] if gpu_count >= 2 else max(GRPO_CONFIG['per_device_train_batch_size'], GRPO_CONFIG['num_generations'])
if gpu_count >= 2:
    full_cmd = (
        "HF_HOME=/workspace/hf-cache NCCL_P2P_DISABLE=1 NCCL_IB_DISABLE=1 "
        "accelerate launch --num_processes 2 --mixed_precision bf16 "
        "scripts/train_finchain_trl_grpo.py "
        f"--model {GRPO_CONFIG['policy_hf_dir']} "
        f"--train-n {GRPO_CONFIG['train_n']} "
        f"--max-steps {GRPO_CONFIG['max_steps']} "
        f"--num-generations {GRPO_CONFIG['num_generations']} "
        f"--max-completion-length {GRPO_CONFIG['max_completion_length']} "
        f"--per-device-train-batch-size {full_per_device_train_batch_size} "
        f"--gradient-accumulation-steps {GRPO_CONFIG['gradient_accumulation_steps']} "
        f"--learning-rate {GRPO_CONFIG['learning_rate']} "
        f"--beta {GRPO_CONFIG['beta_kl']} "
        f"--save-steps {GRPO_CONFIG['save_steps']} "
        f"--output-dir {GRPO_CONFIG['save_dir']} "
        f"--report-to wandb --run-name {GRPO_CONFIG['run_name']}"
    )
    cost_label = "grpo_full_two_gpu"
else:
    full_cmd = (
        "HF_HOME=/workspace/hf-cache python scripts/train_finchain_trl_grpo.py "
        f"--model {GRPO_CONFIG['policy_hf_dir']} "
        f"--train-n {GRPO_CONFIG['train_n']} "
        f"--max-steps {GRPO_CONFIG['max_steps']} "
        f"--num-generations {GRPO_CONFIG['num_generations']} "
        f"--max-completion-length {GRPO_CONFIG['max_completion_length']} "
        f"--per-device-train-batch-size {full_per_device_train_batch_size} "
        f"--gradient-accumulation-steps {GRPO_CONFIG['gradient_accumulation_steps']} "
        f"--learning-rate {GRPO_CONFIG['learning_rate']} "
        f"--beta {GRPO_CONFIG['beta_kl']} "
        f"--save-steps {GRPO_CONFIG['save_steps']} "
        f"--output-dir {GRPO_CONFIG['save_dir']} "
        f"--report-to wandb --run-name {GRPO_CONFIG['run_name']}"
    )
    cost_label = "grpo_full_single_gpu"
    print("WARNING: fewer than 2 CUDA devices visible; falling back to single-GPU GRPO.")

full_start = time.perf_counter()
full_result = run_cmd(full_cmd, check=True)
append_cost_event(
    cost_label,
    elapsed_sec=round(time.perf_counter() - full_start, 1),
    exit_code=full_result.returncode,
)


## Step 5 - Verify checkpoints landed

TRL writes its final checkpoint to `<output_dir>/final`. Intermediate checkpoints land in `<output_dir>/checkpoint-N`.

In [ ]:
from pathlib import Path
import json
import time
from huggingface_hub import snapshot_download

GRPO_CONFIG.setdefault("all_checkpoints_hf_repo", GRPO_CONFIG["hf_candidates_repo"])
GRPO_CONFIG["hf_private"] = False

save_dir = Path(GRPO_CONFIG["save_dir"])

eval_required = {"model.safetensors", "config.json", "tokenizer.json", "tokenizer_config.json"}
full_required = eval_required | {"optimizer.pt", "scheduler.pt", "rng_state.pth", "trainer_state.json", "training_args.bin"}

expected_checkpoint_dirs = [
    save_dir / f"checkpoint-{step}"
    for step in GRPO_CONFIG["grpo_eval_steps"]
]
missing_expected = [ckpt for ckpt in expected_checkpoint_dirs if not ckpt.exists()]
if missing_expected:
    print("restoring GRPO checkpoints from HF:", GRPO_CONFIG["all_checkpoints_hf_repo"])
    print("missing local dirs:", [ckpt.name for ckpt in missing_expected])
    snapshot_download(
        repo_id=GRPO_CONFIG["all_checkpoints_hf_repo"],
        repo_type="model",
        local_dir=str(save_dir),
    )

usable_steps = []
checkpoint_dirs = []
manifest = {"save_dir": str(save_dir), "checkpoints": []}

for ckpt in expected_checkpoint_dirs:
    step = int(ckpt.name.split("-")[-1])
    if not ckpt.exists():
        print(f"{ckpt.name}: missing")
        manifest["checkpoints"].append({
            "name": ckpt.name,
            "step": step,
            "status": "missing",
            "stable_after_5s": False,
            "missing_eval": sorted(eval_required),
            "missing_full": sorted(full_required),
        })
        continue

    before = {str(p.relative_to(ckpt)): p.stat().st_size for p in ckpt.rglob("*") if p.is_file()}
    time.sleep(5)
    after = {str(p.relative_to(ckpt)): p.stat().st_size for p in ckpt.rglob("*") if p.is_file()}

    names = {p.name for p in ckpt.iterdir() if p.is_file()}
    stable = before == after
    eval_ok = stable and eval_required <= names
    full_ok = stable and full_required <= names
    status = "full-resumable" if full_ok else "eval-only" if eval_ok else "incomplete-or-changing"

    print(f"{ckpt.name}: {status}")
    print("  missing_eval:", sorted(eval_required - names))
    print("  missing_full:", sorted(full_required - names))

    manifest["checkpoints"].append({
        "name": ckpt.name,
        "step": step,
        "status": status,
        "stable_after_5s": stable,
        "missing_eval": sorted(eval_required - names),
        "missing_full": sorted(full_required - names),
    })

    if eval_ok:
        usable_steps.append(step)
        checkpoint_dirs.append(ckpt)

GRPO_CONFIG["grpo_eval_steps"] = usable_steps
expected_labels = {f"grpo_checkpoint_{step}" for step in usable_steps}
(save_dir / "CHECKPOINT_MANIFEST.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("usable eval steps:", usable_steps)
print("wrote:", save_dir / "CHECKPOINT_MANIFEST.json")
assert usable_steps, "No eval-usable checkpoints found."

In [ ]:
from getpass import getpass
import json
from huggingface_hub import HfApi, get_token, login, upload_large_folder

save_dir = Path(GRPO_CONFIG["save_dir"])
hf_repo = GRPO_CONFIG["all_checkpoints_hf_repo"]

if get_token() is None:
    login(token=getpass("HF token: "))

api = HfApi()
api.create_repo(repo_id=hf_repo, repo_type="model", private=GRPO_CONFIG["hf_private"], exist_ok=True)

upload_manifest = {
    "repo_id": hf_repo,
    "source_dir": str(save_dir),
    "uploaded_checkpoints": [p.name for p in sorted(save_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))],
    "usable_eval_steps": GRPO_CONFIG["grpo_eval_steps"],
}
(save_dir / "UPLOAD_MANIFEST.json").write_text(json.dumps(upload_manifest, indent=2), encoding="utf-8")

upload_large_folder(
    repo_id=hf_repo,
    repo_type="model",
    folder_path=str(save_dir),
    num_workers=4,
    print_report=True,
)

remote_files = api.list_repo_files(repo_id=hf_repo, repo_type="model")
assert "CHECKPOINT_MANIFEST.json" in remote_files
assert "UPLOAD_MANIFEST.json" in remote_files

for step in GRPO_CONFIG["grpo_eval_steps"]:
    marker = f"checkpoint-{step}/model.safetensors"
    assert marker in remote_files, f"Missing remote model: {marker}"

print("UPLOAD COMPLETE:", hf_repo)
print("remote checkpoint dirs:", sorted({p.split('/')[0] for p in remote_files if p.startswith("checkpoint-")}))

In [ ]:
from huggingface_hub import HfApi

repo = GRPO_CONFIG["all_checkpoints_hf_repo"]
files = HfApi().list_repo_files(repo_id=repo, repo_type="model")

print("repo:", repo)
print("checkpoint dirs:", sorted({p.split("/")[0] for p in files if p.startswith("checkpoint-")}))
print("manifest:", "CHECKPOINT_MANIFEST.json" in files)
print("upload manifest:", "UPLOAD_MANIFEST.json" in files)

for step in GRPO_CONFIG["grpo_eval_steps"]:
    assert f"checkpoint-{step}/model.safetensors" in files
    assert f"checkpoint-{step}/trainer_state.json" in files

print("REMOTE VERIFIED")

## Step 6 - Select GRPO on validation, then evaluate test once

TRL writes HF-format `checkpoint-*` candidates. Score every candidate on the
870-row validation set, require the best GRPO candidate to beat the SFT
baseline on validation, copy only a passing policy to `selected/`, and
compare only that artifact against the SFT baseline on test with ChainEval
enabled.


In [ ]:
import shutil
import torch as _t

grpo_candidate_dirs = {
    f"grpo_{path.name.replace('-', '_')}": path
    for path in checkpoint_dirs
}
if set(grpo_candidate_dirs) != expected_labels:
    raise RuntimeError(
        f"GRPO candidate set mismatch: got {sorted(grpo_candidate_dirs)}, expected {sorted(expected_labels)}"
    )
EVAL_ROOT = PROJECT_ROOT / "results" / "evals" / "grpo_v2_template_disjoint"
VALIDATION_OUT_DIR = EVAL_ROOT / "validation_selection"
TEST_OUT_DIR = EVAL_ROOT / "test_selected_chaineval"
EVAL_CONFIG = {"parallelism": "examples", "batch_size_finchain": 32}
gpus_arg = "0 1" if _t.cuda.is_available() and _t.cuda.device_count() > 1 else "0"
candidate_args = " ".join(f"{label}={path}" for label, path in grpo_candidate_dirs.items())
validation_checkpoint_args = f"sft_baseline={GRPO_CONFIG['policy_hf_dir']} {candidate_args}"

run_cmd(
    "HF_HOME=/workspace/hf-cache python scripts/run_finchain_eval_parallel.py "
    f"--gpus {gpus_arg} --parallelism {EVAL_CONFIG['parallelism']} "
    f"--checkpoints {validation_checkpoint_args} --finchain-split validation --n 870 "
    f"--out-dir {VALIDATION_OUT_DIR} --batch-size-finchain {EVAL_CONFIG['batch_size_finchain']}",
    check=True,
)
validation_summaries = {
    label: json.loads((VALIDATION_OUT_DIR / label / "accuracy_summary.json").read_text(encoding="utf-8"))[0]
    for label in ["sft_baseline", *grpo_candidate_dirs]
}
grpo_validation_summaries = {
    label: validation_summaries[label]
    for label in grpo_candidate_dirs
}

def checkpoint_step(label: str) -> int:
    return int(label.rsplit("_", 1)[-1]) if "checkpoint_" in label else 10**12

selected_label = sorted(
    grpo_validation_summaries,
    key=lambda label: (
        -grpo_validation_summaries[label]["accuracy"],
        -grpo_validation_summaries[label].get("parse_success_rate", 0.0),
        checkpoint_step(label),
    ),
)[0]
sft_validation_accuracy = validation_summaries["sft_baseline"]["accuracy"]
selected_validation_accuracy = grpo_validation_summaries[selected_label]["accuracy"]
if selected_validation_accuracy <= sft_validation_accuracy:
    raise RuntimeError(
        "GRPO failed validation gate: best GRPO checkpoint "
        f"{selected_label} accuracy={selected_validation_accuracy:.6f} did not beat "
        f"sft_baseline accuracy={sft_validation_accuracy:.6f}. Do not run test or push."
    )
grpo_selected_dir = save_dir / "selected"
if grpo_selected_dir.exists():
    shutil.rmtree(grpo_selected_dir)
shutil.copytree(grpo_candidate_dirs[selected_label], grpo_selected_dir)
print("selected on validation:", selected_label, grpo_selected_dir)

selection_metadata = {
    "selection_split": "validation",
    "final_eval_split": "test",
    "selected_checkpoint": str(grpo_selected_dir),
    "selected_label": selected_label,
    "selection_metric": "accuracy",
    "sft_validation_accuracy": sft_validation_accuracy,
    "selected_validation_accuracy": selected_validation_accuracy,
    "validation_summaries": validation_summaries,
    "tie_breaker": "parse_success_rate_then_earliest_step",
    "test_touched_before_selection": False,
}
(EVAL_ROOT / "selection_metadata.json").write_text(
    json.dumps(selection_metadata, indent=2, sort_keys=True),
    encoding="utf-8",
)

run_cmd(
    "HF_HOME=/workspace/hf-cache python scripts/run_finchain_eval_parallel.py "
    f"--gpus {gpus_arg} --parallelism {EVAL_CONFIG['parallelism']} "
    f"--checkpoints sft={GRPO_CONFIG['policy_hf_dir']} grpo_selected={grpo_selected_dir} "
    f"--finchain-split test --n 870 --out-dir {TEST_OUT_DIR} "
    f"--batch-size-finchain {EVAL_CONFIG['batch_size_finchain']} --enable-chaineval",
    check=True,
)
append_cost_event("grpo_v2_selected_test_chaineval", selection=selected_label)


## Step 7 - Headline numbers

The untouched test read is the selected GRPO/RLVR policy versus SFT. Report
final-answer `accuracy` as the primary task metric and ChainEval chain
precision, recall, F1, and alignment fields as reasoning diagnostics.


In [ ]:
BASELINE_AND_SELECTED_LABELS = ["sft", "grpo_selected"]
HEADLINE_METRICS = ["n", "accuracy", "parse_success_rate", "step_recall", "step_precision", "step_f1"]

def compact_metrics(summary):
    return {metric: summary.get(metric) for metric in HEADLINE_METRICS if metric in summary}

test_summaries = {}
for name in BASELINE_AND_SELECTED_LABELS:
    summary_path = TEST_OUT_DIR / name / "accuracy_summary.json"
    test_summaries[name] = json.loads(summary_path.read_text(encoding="utf-8"))[0]
print("validation selection:")
print(json.dumps(validation_summaries, indent=2, sort_keys=True))
print("untouched test summary metrics:")
print(json.dumps({name: compact_metrics(summary) for name, summary in test_summaries.items()}, indent=2, sort_keys=True))
print("untouched test full ChainEval summaries:")
print(json.dumps(test_summaries, indent=2, sort_keys=True))


## Push the GRPO checkpoint to HF Hub

This uploads the HF-format model folder to `shannan-liu1/qwen25-1p5b-finchain-v2-grpo-selected`. Run only after the checkpoint smoke-load/eval cell succeeds and `huggingface-cli whoami` shows the account that can write to `shannan-liu1`.

In [ ]:
from huggingface_hub import HfApi
from huggingface_hub.utils import HfHubHTTPError

api = HfApi()
repo_id = GRPO_CONFIG["hf_selected_repo"]
folder_to_push = Path(grpo_selected_dir)
if not folder_to_push.exists():
    raise FileNotFoundError(f"checkpoint folder does not exist: {folder_to_push}")

api.create_repo(repo_id, exist_ok=True, private=False)
existing_files = [
    path
    for path in api.list_repo_files(repo_id, repo_type="model")
    if path != ".gitattributes"
]
for repo_file in existing_files:
    try:
        api.delete_file(
            path_in_repo=repo_file,
            repo_id=repo_id,
            repo_type="model",
            commit_message=f"Remove stale selected GRPO/RLVR file {repo_file}",
        )
    except HfHubHTTPError as exc:
        print("delete skipped", repo_file, repr(exc))

api.upload_folder(
    folder_path=str(folder_to_push),
    repo_id=repo_id,
    repo_type="model",
    commit_message="FinChain v2 GRPO validation-selected HF checkpoint",
)
print(f"pushed {folder_to_push} to https://huggingface.co/{repo_id}")
append_cost_event("hf_push_grpo_v2_selected", repo_id=repo_id, folder=str(folder_to_push))


## Push GRPO candidates and eval artifacts to HF Hub

Run this after selected-checkpoint upload succeeds. It overwrites only the GRPO candidate/eval paths owned by this notebook.


In [ ]:
from huggingface_hub import HfApi
from huggingface_hub.utils import HfHubHTTPError

api = HfApi()
candidates_repo_id = GRPO_CONFIG["hf_candidates_repo"]
api.create_repo(candidates_repo_id, exist_ok=True, private=False)

for folder in [
    "checkpoint-100",
    "checkpoint-150",
    "checkpoint-200",
    "checkpoint-250",
    "checkpoint-300",
    "checkpoint-400",
    "checkpoint-450",
    "checkpoint-500",
    "final",
    "selected",
    "evals",
]:
    try:
        api.delete_folder(
            path_in_repo=folder,
            repo_id=candidates_repo_id,
            repo_type="model",
            commit_message=f"Remove stale GRPO candidate path {folder}",
        )
        print("deleted stale HF path:", folder)
    except HfHubHTTPError as exc:
        print("delete skipped", folder, repr(exc))

for label, path in grpo_candidate_dirs.items():
    api.upload_folder(
        folder_path=str(path),
        repo_id=candidates_repo_id,
        repo_type="model",
        path_in_repo=path.name,
        commit_message=f"Upload GRPO candidate {path.name}",
    )

api.upload_folder(
    folder_path=str(EVAL_ROOT),
    repo_id=candidates_repo_id,
    repo_type="model",
    path_in_repo="evals/grpo_v2_template_disjoint",
    commit_message="Upload GRPO validation/test eval artifacts",
)

print(f"uploaded GRPO candidates/evals to https://huggingface.co/{candidates_repo_id}")
append_cost_event(
    "hf_push_grpo_v2_candidates_evals",
    repo_id=candidates_repo_id,
    candidates=list(grpo_candidate_dirs),
    eval_root=str(EVAL_ROOT),
)


## Final - Stop the instance

GPU rentals can charge by the hour. After the final eval artifact is saved and no further cells are queued, stop the instance through your provider console or CLI.